# 01 - Daten: VisDrone-DET 2019 vorbereiten

Download, COCO-Konvertierung, SAHI-Slicing (640x640 @ 20% Overlap),
Statistik und auto-generierter Report nach `docs/data_report.md`.

In [ ]:
# Zelle 0 - Imports
import json
import os
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from PIL import Image
import matplotlib.patches as patches

In [ ]:
# Zelle 1 - Setup: Drive + Repo + Dependencies
!pip install -q mmengine pycocotools sahi gdown matplotlib Pillow

from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess

REPO = '/content/ba-mamba-neck'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/raphaelkach/ba-mamba-neck.git', REPO],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, REPO)

In [ ]:
# Zelle 2 - VisDrone herunterladen
RAW = Path('/content/visdrone/raw')
RAW.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'train': '1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn',
    'val':   '1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59',
}

for split, file_id in SPLITS.items():
    zip_path = RAW / f'VisDrone2019-DET-{split}.zip'
    extract_dir = RAW / f'VisDrone2019-DET-{split}'

    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f'{split}: bereits entpackt, skip')
        continue

    print(f'{split}: download ...')
    !gdown --fuzzy "https://drive.google.com/file/d/{file_id}/view" -O "{zip_path}"

    assert zip_path.exists() and zip_path.stat().st_size > 1_000_000, (
        f'{split}: Download fehlgeschlagen. Manuell von '
        f'https://github.com/VisDrone/VisDrone-Dataset herunterladen '
        f'und als {zip_path} ablegen.'
    )

    print(f'{split}: entpacken ...')
    !unzip -q -o "{zip_path}" -d "{RAW}"

print("Done")

In [ ]:
# Zelle 3 - VisDrone konvertieren + SAHI-Slicing
!python data/prepare.py --output /content/visdrone

In [ ]:
# Zelle 4 - 5 Beispielbilder aus dem Val-Split mit Bounding Boxes
import random

# Zelle 4 - 5 Beispielbilder aus dem Val-Split mit Bounding Boxes
ROOT = Path('/content/visdrone')
VAL_JSON = ROOT / 'annotations' / 'val_unsliced.json'
VAL_IMAGES = ROOT / 'raw' / 'VisDrone2019-DET-val' / 'images'

coco = json.loads(VAL_JSON.read_text())
id_to_name = {c['id']: c['name'] for c in coco['categories']}
ann_by_img = {}
for a in coco['annotations']:
    ann_by_img.setdefault(a['image_id'], []).append(a)

random.seed(42)
samples = random.sample(coco['images'], k=5)

fig, axes = plt.subplots(5, 1, figsize=(12, 30))
for ax, img_info in zip(axes, samples):
    img = Image.open(VAL_IMAGES / img_info['file_name'])
    ax.imshow(img)
    for ann in ann_by_img.get(img_info['id'], []):
        x, y, w, h = ann['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=1,
                                       edgecolor='lime', facecolor='none'))
        ax.text(x, max(0, y - 2), id_to_name[ann['category_id']],
                color='lime', fontsize=6)
    ax.set_title(f"{img_info['file_name']} - {len(ann_by_img.get(img_info['id'], []))} objs")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Zelle 5 - Datenanalyse (publication-ready, PhD-Level)
# --- PhD-Level Style ---
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.6,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.titlepad': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'lines.linewidth': 1.2,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.15,
})

# Colorblind-safe Palette (Wong 2011, Nature Methods)
C_SMALL  = '#E69F00'  # orange
C_MEDIUM = '#56B4E9'  # sky blue
C_LARGE  = '#009E73'  # bluish green
C_HIST   = '#0072B2'  # blue
C_ACCENT = '#D55E00'  # vermillion
C_TEXT   = '#333333'

ROOT = Path('/content/visdrone')
Path('results/figures').mkdir(parents=True, exist_ok=True)

splits = {
    'Train (original)':   ROOT / 'annotations' / 'train_unsliced.json',
    'Train (SAHI 640\u00b2)':  ROOT / 'annotations' / 'train_sliced.json',
    'Val (original)':     ROOT / 'annotations' / 'val_unsliced.json',
    'Val (SAHI 640\u00b2)':    ROOT / 'annotations' / 'val_sliced.json',
}

def load_sides(path):
    coco = json.loads(path.read_text())
    areas = np.array([a['area'] for a in coco['annotations']])
    return np.sqrt(areas)

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# Figure 1: Gr\u00f6\u00dfenverteilung \u2014 4-Panel Histogramm
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for ax, (label, path) in zip(axes.flat, splits.items()):
    sides = load_sides(path)
    n = len(sides)
    s = np.sum(sides < 32)
    m = np.sum((sides >= 32) & (sides < 96))
    l = np.sum(sides >= 96)

    ax.hist(sides, bins=100, color=C_HIST, alpha=0.75, linewidth=0)
    ymax = ax.get_ylim()[1]

    # Schwellenlinien mit Direct Labels
    for thresh, clr, lbl, count in [
        (32, C_SMALL, 'small', s),
        (96, C_ACCENT, 'large', l),
    ]:
        ax.axvline(thresh, color=clr, ls='-', lw=1.2, alpha=0.8)
        ax.text(thresh + 3, ymax * 0.92, f'{thresh}px',
                color=clr, fontsize=9, fontweight='bold', va='top')

    # Prozent-Annotation oben links
    ax.text(0.03, 0.95,
            f'S {100*s/n:.0f}%  \u00b7  M {100*m/n:.0f}%  \u00b7  L {100*l/n:.0f}%',
            transform=ax.transAxes, fontsize=9, color=C_TEXT,
            va='top', fontfamily='monospace')

    ax.set_title(f'{label}   n = {n:,}')
    ax.set_xlim(0, 250)

axes[1, 0].set_xlabel('\u221aarea (px)')
axes[1, 1].set_xlabel('\u221aarea (px)')
axes[0, 0].set_ylabel('Annotations')
axes[1, 0].set_ylabel('Annotations')

for ax in axes.flat:
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.tight_layout(h_pad=2.5, w_pad=2.0)
fig.savefig('results/figures/size_distribution.pdf')
plt.show()
print('-> results/figures/size_distribution.pdf')


# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# Figure 2: SAHI-Effekt \u2014 Grouped Bar (vorher/nachher)
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(3)  # S, M, L
w = 0.18
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]

bar_labels = ['Train original', 'Train SAHI', 'Val original', 'Val SAHI']
all_paths = [
    splits['Train (original)'], splits['Train (SAHI 640\u00b2)'],
    splits['Val (original)'], splits['Val (SAHI 640\u00b2)'],
]
bar_colors = [C_HIST, '#5DA5DA', C_ACCENT, '#FAA43A']

for i, (lbl, path) in enumerate(zip(bar_labels, all_paths)):
    sides = load_sides(path)
    n = len(sides)
    pcts = [
        100 * np.sum(sides < 32) / n,
        100 * np.sum((sides >= 32) & (sides < 96)) / n,
        100 * np.sum(sides >= 96) / n,
    ]
    bars = ax.bar(x + offsets[i], pcts, w, label=lbl,
                  color=bar_colors[i], alpha=0.82, linewidth=0)
    # Direct Labels \u00fcber jedem Balken
    for bar, pct in zip(bars, pcts):
        if pct > 3:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{pct:.0f}', ha='center', va='bottom', fontsize=8, color=C_TEXT)

ax.set_xticks(x)
ax.set_xticklabels(['Small\n(< 32\u00b2 px)', 'Medium\n(32\u00b2\u201396\u00b2 px)', 'Large\n(> 96\u00b2 px)'])
ax.set_ylabel('Anteil (%)')
ax.set_ylim(0, 100)
ax.legend(fontsize=9, frameon=False, ncol=2, loc='upper right')

fig.tight_layout()
fig.savefig('results/figures/sahi_effect.pdf')
plt.show()
print('-> results/figures/sahi_effect.pdf')


# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# Figure 3: Klassenverteilung \u2014 Horizontal Bar mit Direct Labels
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
fig, ax = plt.subplots(figsize=(8, 4.5))

coco_train = json.loads((ROOT / 'annotations' / 'train_unsliced.json').read_text())
id_to_name = {c['id']: c['name'] for c in coco_train['categories']}
cls_counts = defaultdict(int)
for a in coco_train['annotations']:
    cls_counts[id_to_name[a['category_id']]] += 1

sorted_cls = sorted(cls_counts.items(), key=lambda x: x[1])
names = [c[0] for c in sorted_cls]
counts = [c[1] for c in sorted_cls]
max_count = max(counts)

# Einheitliche Farbe, kein Regenbogen
bars = ax.barh(names, counts, color=C_HIST, alpha=0.8, height=0.65, linewidth=0)

# Direct Labels rechts neben den Balken
for bar, count in zip(bars, counts):
    ax.text(count + max_count * 0.015,
            bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=9, color=C_TEXT)

ax.set_xlabel('Annotations (train, unsliced)')
ax.set_xlim(0, max_count * 1.15)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.tight_layout()
fig.savefig('results/figures/class_distribution.pdf')
plt.show()
print('-> results/figures/class_distribution.pdf')

In [ ]:
# Zelle 6 - docs/data_report.md generieren (Skript ist single source of truth)
!PYTHONPATH=. python scripts/generate_data_report.py --visdrone /content/visdrone